# Module 3: Temporal Difference Learning
## Notebook 3: Algorithm Comparison — TD(0), SARSA, Q-Learning, Expected SARSA

**Learning Objectives**

By completing this notebook, you will:
1. Implement Expected SARSA and understand its variance-reduction advantage over SARSA
2. Run a statistically rigorous comparison of four TD algorithms across multiple seeds
3. Interpret learning curves with confidence bands and identify sample efficiency differences
4. Select the right algorithm for a given deployment scenario

**Prerequisites**
- TD(0) prediction (Notebook 1)
- SARSA and Q-learning (Notebook 2)
- Understanding of on-policy vs. off-policy distinction

**Estimated Time: 13 minutes**

---

### The Four Algorithms at a Glance

| Algorithm | Update target | On/Off-policy | Key property |
|-----------|--------------|---------------|--------------|
| TD(0) | $R + \gamma V(S')$ | On-policy | Prediction only |
| SARSA | $R + \gamma Q(S', A')$ | On-policy | Uses actual next action |
| Q-learning | $R + \gamma \max_{a'} Q(S', a')$ | Off-policy | Targets optimal policy |
| Expected SARSA | $R + \gamma \mathbb{E}_{\pi}[Q(S', a')]$ | On or off-policy | Lower variance than SARSA |

In [ ]:
learning_objectives([
    "By completing this notebook, you will:",
    "- TD(0) prediction (Notebook 1)",
    "SARSA and Q-learning (Notebook 2)",
    "Understanding of on-policy vs. off-policy distinction",
    "---"
])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Import dependencies and define global settings
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print("Setup complete.")

## 1. Environment and Shared Utilities

We benchmark on Cliff Walking (same as Notebook 2) because all four algorithms can be applied to it and the ground truth optimal return (-13) is known.

In [ ]:
# Purpose: Define CliffWalking and shared helper functions
# Key Concept: Reusing the same environment makes comparisons fair

class CliffWalking:
    ROWS = 4
    COLS = 12
    N_ACTIONS = 4
    ACTION_DELTAS = {0: (-1,0), 1: (0,1), 2: (1,0), 3: (0,-1)}

    def __init__(self):
        self.start = (self.ROWS-1, 0)
        self.goal  = (self.ROWS-1, self.COLS-1)
        self.cliff = {(self.ROWS-1, c) for c in range(1, self.COLS-1)}

    def reset(self):
        return self.start

    def step(self, state, action):
        dr, dc = self.ACTION_DELTAS[action]
        r, c   = state
        nr = max(0, min(self.ROWS-1, r+dr))
        nc = max(0, min(self.COLS-1, c+dc))
        ns = (nr, nc)
        if ns in self.cliff:
            return self.start, -100.0, False
        elif ns == self.goal:
            return ns, -1.0, True
        else:
            return ns, -1.0, False

    def state_idx(self, state):
        return state[0] * self.COLS + state[1]

    @property
    def n_states(self):
        return self.ROWS * self.COLS


def epsilon_greedy(Q_row, epsilon, n_actions, rng):
    """Epsilon-greedy action selection."""
    if rng.random() < epsilon:
        return int(rng.integers(n_actions))
    return int(np.argmax(Q_row))


env = CliffWalking()
OPTIMAL_RETURN = -13  # minimum steps along the bottom = 11 steps + 0 cliff hits
print(f"Environment: {env.ROWS}x{env.COLS} Cliff Walking")
print(f"Known optimal return: {OPTIMAL_RETURN}")

## 2. Expected SARSA

Expected SARSA replaces the sampled next action with the **expected Q-value** under the current policy:

$$Q(S, A) \leftarrow Q(S, A) + \alpha \left[ R + \gamma \sum_{a'} \pi(a' | S') Q(S', a') - Q(S, A) \right]$$

For epsilon-greedy policy with $n$ actions:
$$\sum_{a'} \pi(a'|s') Q(s', a') = \frac{\epsilon}{n} \sum_{a'} Q(s', a') + (1 - \epsilon) \max_{a'} Q(s', a')$$

This eliminates the **variance from action sampling**, making Expected SARSA generally more stable than SARSA.

In [ ]:
# Purpose: Implement all four TD control algorithms in a unified interface
# Key Concept: Parameterizing by 'mode' makes fair comparison easy

def td_control(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1,
               mode='sarsa', seed=42):
    """
    Unified TD control function supporting SARSA, Q-learning, and Expected SARSA.

    Parameters
    ----------
    mode : str — 'sarsa', 'qlearning', or 'expected_sarsa'

    Returns
    -------
    Q              : np.ndarray, shape (n_states, n_actions)
    episode_rewards: np.ndarray, shape (n_episodes,)
    """
    rng = np.random.default_rng(seed)
    Q   = np.zeros((env.n_states, env.N_ACTIONS))
    rewards = np.zeros(n_episodes)

    for ep in range(n_episodes):
        state = env.reset()
        s_idx = env.state_idx(state)
        total = 0.0
        done  = False

        # SARSA needs the first action selected before the loop
        if mode == 'sarsa':
            action = epsilon_greedy(Q[s_idx], epsilon, env.N_ACTIONS, rng)

        while not done:
            if mode != 'sarsa':
                action = epsilon_greedy(Q[s_idx], epsilon, env.N_ACTIONS, rng)

            next_state, reward, done = env.step(state, action)
            ns_idx = env.state_idx(next_state)
            total += reward

            # Compute target based on algorithm mode
            if mode == 'sarsa':
                # On-policy: use actual next action
                next_action = epsilon_greedy(Q[ns_idx], epsilon, env.N_ACTIONS, rng)
                target = reward + gamma * Q[ns_idx, next_action]

            elif mode == 'qlearning':
                # Off-policy: use greedy next action
                target = reward + gamma * np.max(Q[ns_idx])

            elif mode == 'expected_sarsa':
                # Compute expected value under epsilon-greedy
                n = env.N_ACTIONS
                greedy_val = np.max(Q[ns_idx])
                avg_val    = np.mean(Q[ns_idx])
                # Epsilon-greedy expected value:
                # (1-eps)*greedy + eps*average (uniform exploration)
                expected_q = (1 - epsilon) * greedy_val + epsilon * avg_val
                target = reward + gamma * expected_q

            else:
                raise ValueError(f"Unknown mode: {mode}")

            # TD update
            Q[s_idx, action] += alpha * (target - Q[s_idx, action])

            state  = next_state
            s_idx  = ns_idx
            if mode == 'sarsa':
                action = next_action

        rewards[ep] = total

    return Q, rewards


# Quick test of each mode
for mode in ['sarsa', 'qlearning', 'expected_sarsa']:
    _, r = td_control(env, n_episodes=200, mode=mode, seed=0)
    print(f"{mode:15s}: mean last-50 reward = {np.mean(r[-50:]):.1f}")

## 3. Multi-Run Benchmark

A single run is noisy. We run each algorithm 30 times with different seeds, then plot the **mean ± 1 standard deviation** of episode rewards. This shows both average performance and variability.

In [ ]:
# Purpose: Run each algorithm 30 times and collect reward histories
# Key Concept: Multiple runs reveal variance, not just mean performance

N_RUNS     = 30
N_EPISODES = 300
ALPHA      = 0.5
EPSILON    = 0.1

ALGORITHMS = {
    'SARSA':          {'mode': 'sarsa',          'color': 'steelblue'},
    'Q-learning':     {'mode': 'qlearning',       'color': 'tomato'},
    'Expected SARSA': {'mode': 'expected_sarsa',  'color': 'forestgreen'},
}

results = {}   # algo_name -> np.ndarray shape (N_RUNS, N_EPISODES)

for name, cfg in ALGORITHMS.items():
    run_rewards = np.zeros((N_RUNS, N_EPISODES))
    for run in range(N_RUNS):
        _, r = td_control(env, N_EPISODES,
                          alpha=ALPHA, epsilon=EPSILON,
                          mode=cfg['mode'], seed=run)
        run_rewards[run] = r
    results[name] = run_rewards
    final_mean = np.mean(run_rewards[:, -50:].mean(axis=1))
    print(f"{name:15s}: final avg reward = {final_mean:.1f}")

print("\nAll runs complete.")

## 4. Learning Curves with Confidence Bands

In [ ]:
# Purpose: Plot smoothed learning curves with ±1 std bands for each algorithm
# Key Concept: Confidence bands show both speed and stability of learning

def smooth(arr, window=15):
    """Moving average along episodes axis."""
    kernel = np.ones(window) / window
    return np.array([np.convolve(row, kernel, mode='valid') for row in arr])


fig, ax = plt.subplots(figsize=(13, 6))

for name, cfg in ALGORITHMS.items():
    raw   = results[name]                  # (N_RUNS, N_EPISODES)
    smth  = smooth(raw, window=15)         # (N_RUNS, N_EPISODES-14)
    mean  = smth.mean(axis=0)
    std   = smth.std(axis=0)
    x     = np.arange(len(mean))
    color = cfg['color']

    ax.plot(x, mean, color=color, linewidth=2, label=name)
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.15)

ax.axhline(OPTIMAL_RETURN, color='black', linestyle='--', linewidth=1.5,
           label=f'Optimal return ({OPTIMAL_RETURN})')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Episode Reward (smoothed)', fontsize=12)
ax.set_title(
    f'TD Control Algorithms on Cliff Walking\n'
    f'({N_RUNS} runs, alpha={ALPHA}, epsilon={EPSILON})',
    fontsize=13
)
ax.legend(fontsize=11)
ax.set_ylim(-300, 5)

plt.tight_layout()
plt.savefig('../resources/algorithm_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Summary Statistics Table

In [ ]:
# Purpose: Compute and display summary statistics for each algorithm
# Key Concept: Quantitative comparison complements visual inspection

print(f"{'Algorithm':20s} {'Final Mean':>12} {'Final Std':>10} {'Episodes to -50':>16}")
print("-" * 62)

for name in ALGORITHMS:
    raw = results[name]  # (N_RUNS, N_EPISODES)

    # Final performance: mean reward of last 30 episodes, averaged over runs
    final_per_run = raw[:, -30:].mean(axis=1)   # shape (N_RUNS,)
    final_mean    = final_per_run.mean()
    final_std     = final_per_run.std()

    # Episodes to first achieve mean reward > -50 (over a 10-ep window)
    mean_curve = raw.mean(axis=0)  # shape (N_EPISODES,)
    threshold_ep = N_EPISODES      # default: never reached
    for ep in range(10, N_EPISODES):
        if mean_curve[ep-10:ep].mean() > -50:
            threshold_ep = ep
            break

    threshold_str = str(threshold_ep) if threshold_ep < N_EPISODES else "not reached"
    print(f"{name:20s} {final_mean:>12.1f} {final_std:>10.1f} {threshold_str:>16}")

### Exercise 3.3.1: Sweep Over Alpha Values

**Task:** For Expected SARSA and Q-learning, sweep alpha over [0.1, 0.3, 0.5, 0.7] and plot the mean final performance (last 30 episodes, 20 runs) for each alpha. Which algorithm is more sensitive to the choice of alpha?

**Expected Output:** A line plot with one line per algorithm, x-axis = alpha, y-axis = mean final reward.

<details>
<summary>Hint 1</summary>
Use the td_control function with mode='expected_sarsa' and mode='qlearning'. Collect np.mean(rewards[-30:]) over 20 runs for each alpha.
</details>

<details>
<summary>Hint 2</summary>
Expected SARSA tends to be more robust to larger alpha because it eliminates action-sampling variance. You should see Q-learning degrade faster at high alpha.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

alphas    = [0.1, 0.3, 0.5, 0.7]
n_runs_ex = 20
n_ep_ex   = 250

modes_to_test = {
    'Q-learning':     ('qlearning',      'tomato'),
    'Expected SARSA': ('expected_sarsa',  'forestgreen'),
}

alpha_results = {name: [] for name in modes_to_test}

for alpha in alphas:
    for name, (mode, _) in modes_to_test.items():
        run_finals = []
        for run in range(n_runs_ex):
            _, r = td_control(env, n_ep_ex, alpha=alpha, epsilon=0.1,
                              mode=mode, seed=run)
            run_finals.append(np.mean(r[-30:]))
        alpha_results[name].append(np.mean(run_finals))

fig, ax = plt.subplots(figsize=(9, 5))
for name, (_, color) in modes_to_test.items():
    ax.plot(alphas, alpha_results[name], 'o-', color=color,
            linewidth=2, markersize=8, label=name)

ax.set_xlabel('Alpha (learning rate)', fontsize=12)
ax.set_ylabel('Mean Final Reward (last 30 eps, 20 runs)', fontsize=12)
ax.set_title('Sensitivity to Alpha: Q-learning vs. Expected SARSA', fontsize=13)
ax.legend(fontsize=11)
ax.set_xticks(alphas)
plt.tight_layout()
plt.show()

for name in modes_to_test:
    print(f"{name}: {dict(zip(alphas, [round(v,1) for v in alpha_results[name]]))}")

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_331():
    for name in modes_to_test:
        assert len(alpha_results[name]) == len(alphas), \
            f"{name}: expected {len(alphas)} values, got {len(alpha_results[name])}"
    # Expected SARSA should beat Q-learning at high alpha
    idx_high = alphas.index(0.7)
    es_high = alpha_results['Expected SARSA'][idx_high]
    ql_high = alpha_results['Q-learning'][idx_high]
    assert es_high >= ql_high, (
        f"At alpha=0.7, Expected SARSA ({es_high:.1f}) should be >= Q-learning ({ql_high:.1f}). "
        "Expected SARSA is more robust to large alpha."
    )
    print("Exercise 3.3.1 passed!")
    print(f"At alpha=0.7 — Expected SARSA: {es_high:.1f}, Q-learning: {ql_high:.1f}")

test_exercise_331()

### Exercise 3.3.2: Epsilon Decay Strategy

**Task:** Implement a linearly decaying epsilon schedule (epsilon decays from 1.0 to 0.01 over the first 200 episodes, then stays at 0.01). Apply it to Q-learning on Cliff Walking for 300 episodes. Compare the learning curve to constant epsilon=0.1 Q-learning.

**Expected Output:** Two Q-learning learning curves (averaged over 20 runs) showing that epsilon decay improves final performance.

<details>
<summary>Hint 1</summary>
Modify td_control to accept an epsilon_schedule callable: epsilon = epsilon_schedule(episode_number). Or write a new function that recomputes epsilon each episode.
</details>

<details>
<summary>Hint 2</summary>
Linear decay: epsilon(ep) = max(0.01, 1.0 - (1.0 - 0.01) * ep / 200)
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def q_learning_with_decay(env, n_episodes, alpha=0.5, gamma=1.0,
                           eps_start=1.0, eps_end=0.01,
                           decay_episodes=200, seed=42):
    """
    Q-learning with linearly decaying epsilon.

    Returns
    -------
    Q       : np.ndarray
    rewards : np.ndarray, shape (n_episodes,)
    """
    rng = np.random.default_rng(seed)
    Q   = np.zeros((env.n_states, env.N_ACTIONS))
    rewards = np.zeros(n_episodes)

    for ep in range(n_episodes):
        # Linear decay: from eps_start to eps_end over decay_episodes
        t       = min(ep, decay_episodes)
        epsilon = eps_start + (eps_end - eps_start) * t / decay_episodes

        state = env.reset()
        s_idx = env.state_idx(state)
        total = 0.0
        done  = False

        while not done:
            action = epsilon_greedy(Q[s_idx], epsilon, env.N_ACTIONS, rng)
            next_state, reward, done = env.step(state, action)
            ns_idx = env.state_idx(next_state)
            total += reward
            target = reward + gamma * np.max(Q[ns_idx])
            Q[s_idx, action] += alpha * (target - Q[s_idx, action])
            state = next_state
            s_idx = ns_idx

        rewards[ep] = total

    return Q, rewards


# Collect results over 20 runs
N_RUNS_EX = 20
N_EP_EX   = 300

decay_rewards = np.zeros((N_RUNS_EX, N_EP_EX))
const_rewards = np.zeros((N_RUNS_EX, N_EP_EX))

for run in range(N_RUNS_EX):
    _, decay_rewards[run] = q_learning_with_decay(env, N_EP_EX, alpha=0.5, seed=run)
    _, const_rewards[run] = td_control(env, N_EP_EX, alpha=0.5, epsilon=0.1,
                                       mode='qlearning', seed=run)

# Plot
fig, ax = plt.subplots(figsize=(11, 5))
for arr, label, color in [
    (decay_rewards, 'Q-learning (epsilon decay)', 'purple'),
    (const_rewards, 'Q-learning (epsilon=0.1)',    'tomato'),
]:
    mean = arr.mean(axis=0)
    std  = arr.std(axis=0)
    ax.plot(mean, color=color, linewidth=2, label=label)
    ax.fill_between(np.arange(N_EP_EX), mean-std, mean+std, color=color, alpha=0.15)

ax.axhline(OPTIMAL_RETURN, color='black', linestyle='--', linewidth=1.5,
           label=f'Optimal ({OPTIMAL_RETURN})')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Episode Reward', fontsize=12)
ax.set_title(f'Q-learning: Epsilon Decay vs. Constant Epsilon ({N_RUNS_EX} runs)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(-300, 5)
plt.tight_layout()
plt.show()

print(f"Decay — final mean: {np.mean(decay_rewards[:, -30:]):.1f}")
print(f"Const — final mean: {np.mean(const_rewards[:, -30:]):.1f}")

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_332():
    assert decay_rewards.shape == (N_RUNS_EX, N_EP_EX), \
        f"decay_rewards shape {decay_rewards.shape} != ({N_RUNS_EX}, {N_EP_EX})"
    final_decay = np.mean(decay_rewards[:, -30:])
    final_const = np.mean(const_rewards[:, -30:])
    assert final_decay >= final_const, (
        f"Decayed epsilon Q-learning ({final_decay:.1f}) should achieve higher final reward "
        f"than constant epsilon ({final_const:.1f})"
    )
    print("Exercise 3.3.2 passed!")
    print(f"Epsilon decay improvement: {final_decay - final_const:.1f} reward units")

test_exercise_332()

In [ ]:
section_divider("Key Takeaways")

## Key Takeaways

1. **Expected SARSA dominates SARSA** in most settings: removing action-sampling variance reduces noise in updates, enabling larger learning rates and faster convergence.
2. **Q-learning learns the optimal policy** but pays for exploration during training. In safety-critical domains, this exploration cost is unacceptable.
3. **Confidence bands reveal stability**: a lower mean with tighter std may be more desirable in practice than a higher mean with high variance.
4. **Alpha sensitivity**: Expected SARSA tolerates higher alpha because it eliminates variance from action sampling; Q-learning's convergence degrades at alpha > 0.5.
5. **Epsilon decay** bridges exploration and exploitation: starting high encourages early exploration, decaying toward zero locks in the learned policy.

## What's Next

Module 4 moves beyond tabular methods. When the state space is continuous (as in Mountain Car), we can't store a Q-value for every state. **Function approximation** — starting with tile coding and linear TD — lets us generalize across states. This is the bridge from classical RL to deep RL.

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])